# MAUDE pipeline walkthrough

Loads the tables written by `python -m src.run` and rebuilds the figures.
Nothing here re-downloads the corpus — this reads the pipeline's output.

    python -m src.run --config configs/shti2025_ovesco_etl_llm.yaml

In [ ]:
import sys, pathlib
import pandas as pd

sys.path.insert(0, "..")
from src.config import StudyConfig
from src.visualize import problem_flows, sankey, treemap

STUDY = "shti2025_ovesco_etl_llm"
PROCESSED = pathlib.Path("../data/processed")

cfg = StudyConfig.from_yaml(f"../configs/{STUDY}.yaml")
print(cfg.name)
print(cfg.citation)

## 1. Cohort size vs the published number

The first thing to check on any re-run.

In [ ]:
flat = pd.read_csv(PROCESSED / f"{STUDY}_flat.csv")
print(f"cohort:   {len(flat)}")
print(f"paper:    {cfg.expected_reports}")
print("MATCH" if len(flat) == cfg.expected_reports else f"DIFFERS by {len(flat) - cfg.expected_reports:+d}")

## 2. Category counts vs Table 2

Long-form tables, so a category containing a comma stays one category.
Compare these against the paper before trusting any figure built on them.

In [ ]:
prod = pd.read_csv(PROCESSED / f"{STUDY}_product_problems.csv")
pat  = pd.read_csv(PROCESSED / f"{STUDY}_patient_problems.csv")

print(f"device problem mentions : {len(prod):4d}   ({prod['product_problem'].nunique()} distinct)")
print(f"patient event mentions  : {len(pat):4d}   ({pat['patient_problem'].nunique()} distinct)")
prod["product_problem"].value_counts()

## 3. Sankey — device problems to patient outcomes

In [ ]:
flows = problem_flows(prod, pat)
sankey(flows, f"{STUDY}: device problems to patient outcomes")

## 4. Treemap — procedural context from the narratives

Requires the LLM stage to have run (`--stages llm`).

In [ ]:
narr = PROCESSED / f"{STUDY}_narrative_extraction.csv"
uc   = PROCESSED / f"{STUDY}_use_case.csv"

if narr.exists() and uc.exists():
    merged = pd.read_csv(narr).merge(pd.read_csv(uc), on="mdr_report_key", how="inner")
    display(treemap(merged, ["event_timing", "use_case"], f"{STUDY}: use case by event timing"))
else:
    print("LLM outputs not found — run: python -m src.run --config ... --stages llm")

## 5. What the LLM stage failed on

Failures are written out rather than dropped. An empty frame here means every
report was processed; a non-empty one tells you exactly which were not.

In [ ]:
fails = PROCESSED / f"{STUDY}_narrative_extraction_failures.csv"
print("no failures" if not fails.exists() else pd.read_csv(fails).to_string())